# Indian Real Estate Analysis & Machine Learning Project
---


This notebook includes:

- Data Cleaning
- Missing Value Handling
- Encoding
- Outlier Detection
- Feature Engineering
- Exploratory Data Analysis (EDA)
- Machine Learning Price Prediction
- Investment Analysis
- Recommendation System

Dataset: Indian Real Estate Dataset


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import NearestNeighbors

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10,6)


In [ ]:

# Load Dataset
df = pd.read_csv('indian_realestate_dataset_1000.csv')

# Display first rows
df.head()


In [ ]:

# Dataset Overview

print("Shape:", df.shape)
print("\nColumns:\n", df.columns)
print("\nInfo:")
df.info()

print("\nMissing Values:\n")
print(df.isnull().sum())


In [ ]:

# Handle Missing Values

for col in df.select_dtypes(include=['object']).columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

for col in df.select_dtypes(include=['int64', 'float64']).columns:
    df[col].fillna(df[col].median(), inplace=True)

print(df.isnull().sum())


In [ ]:

# Encoding Categorical Features

label_encoders = {}

for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

df.head()


In [ ]:

# Outlier Detection using IQR

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

print("Shape after removing outliers:", df.shape)


In [ ]:

# Feature Engineering

if 'monthly_rent' in df.columns and 'price' in df.columns:
    df['rental_yield'] = (df['monthly_rent'] * 12 / df['price']) * 100

if 'built_up_area' in df.columns and 'price' in df.columns:
    df['price_per_sqft_calc'] = df['price'] / df['built_up_area']

df.head()


In [ ]:

# Correlation Heatmap

plt.figure(figsize=(14,10))
sns.heatmap(df.corr(), cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()


In [ ]:

# City-wise Average Price

if 'city' in df.columns and 'price' in df.columns:
    city_price = df.groupby('city')['price'].mean().sort_values(ascending=False)

    city_price.plot(kind='bar')
    plt.title("City-wise Average Price")
    plt.ylabel("Average Price")
    plt.show()


In [ ]:

# Property Type Distribution

if 'property_type' in df.columns:
    sns.countplot(x=df['property_type'])
    plt.title("Property Type Distribution")
    plt.xticks(rotation=45)
    plt.show()


In [ ]:

# Rent vs Price

if 'monthly_rent' in df.columns and 'price' in df.columns:
    sns.scatterplot(x=df['monthly_rent'], y=df['price'])
    plt.title("Rent vs Property Price")
    plt.show()


In [ ]:

# Investment Analysis

if 'rental_yield' in df.columns and 'city' in df.columns:
    best_roi = df.groupby('city')['rental_yield'].mean().sort_values(ascending=False)

    print("Best ROI Cities:")
    print(best_roi.head(10))

    best_roi.head(10).plot(kind='bar')
    plt.title("Top ROI Cities")
    plt.ylabel("Rental Yield")
    plt.show()


In [ ]:

# Appreciation Trend Analysis

if 'annual_appreciation' in df.columns:
    sns.histplot(df['annual_appreciation'], kde=True)
    plt.title("Annual Appreciation Distribution")
    plt.show()


In [ ]:

# Machine Learning Price Prediction

target_col = 'price'

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)


In [ ]:

# Recommendation System

features = df.drop(columns=['price'])

model_knn = NearestNeighbors(n_neighbors=5, metric='euclidean')
model_knn.fit(features)

sample_index = 0

distances, indices = model_knn.kneighbors([features.iloc[sample_index]])

print("Recommended Similar Properties:\n")
print(df.iloc[indices[0]])



# Conclusion

This notebook successfully demonstrates:

- Data Cleaning
- Encoding
- Outlier Detection
- Feature Engineering
- Exploratory Data Analysis
- Machine Learning Prediction
- Investment Analysis
- Recommendation System

This project can be extended into:
- Streamlit Web App
- Real Estate Dashboard
- AI Property Assistant
- Advanced ML Models
